# Test: does the analysis reproduce from the raw profiles?

Checks `SLG1171_DASTool_bins_9.fa_k141_216652:14297`, allele **T**, in the **HF PRE**
mice — the one stratum where the answer turns on an *absence* (zero T reads in all
15 mice). A miscount there would turn a rare pre-existing variant into a de novo
mutation, so it is the number worth testing.

Independent in the way that matters: it re-reads the raw profiles and reuses none
of `pre_allele_presence.ipynb`'s derived objects — not `tidy`, not the roster
merge, not the zero-fill. Passes if the totals match section 4.

In [1]:
import pandas as pd
from pathlib import Path

RUN = Path("/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_mapq20/longitudinal")
MAG = "SLG1171_DASTool_bins_9"
CONTIG, POSITION, ALLELE = f"{MAG}.fa_k141_216652", 14297, "T"

# What section 4 reports for HF PRE at this site.
EXPECTED = {"reads": 0, "depth": 159, "mice_with_allele": 0}

# The 15 HF PRE samples, with the path to each one's own profile.
meta = pd.read_csv(RUN / "inputMetadata" / "inputMetadata_pre_end-fat_control"
                   / f"{MAG}_metadata.tsv", sep="\t")
meta = meta[(meta.group == "fat") & (meta.time == "pre")]

rows = []
for s in meta.itertuples(index=False):
    p = pd.read_csv(s.file_path, sep="\t", usecols=["contig", "position", "total_coverage", ALLELE])
    hit = p[(p.contig == CONTIG) & (p.position == POSITION)]
    # No row means no high-quality coverage at this position -- count it as zero.
    rows.append({"sample_id": s.sample_id,
                 "reads": int(hit[ALLELE].iloc[0]) if len(hit) else 0,
                 "depth": int(hit.total_coverage.iloc[0]) if len(hit) else 0})

obs = pd.DataFrame(rows)
got = {"reads": int(obs.reads.sum()), "depth": int(obs.depth.sum()),
       "mice_with_allele": int((obs.reads > 0).sum())}

print(f"{CONTIG}:{POSITION}   allele {ALLELE}   HF PRE\n")
print(obs.to_string(index=False))
print(f"\nexpected: {EXPECTED}")
print(f"got:      {got}")
assert got == EXPECTED, "independent scan disagrees with the analysis"
print("\nPASS - section 4 reproduces from the raw profiles.")

SLG1171_DASTool_bins_9.fa_k141_216652:14297   allele T   HF PRE

sample_id  reads  depth
   SLG193      0      6
   SLG194      0     17
  SLG328B      0     10
   SLG423      0      4
   SLG424      0     15
   SLG443      0     23
   SLG444      0      7
   SLG595      0     13
   SLG596      0     11
   SLG615      0      1
   SLG617      0      5
   SLG888      0      4
   SLG889      0     12
   SLG943      0      9
   SLG944      0     22

expected: {'reads': 0, 'depth': 159, 'mice_with_allele': 0}
got:      {'reads': 0, 'depth': 159, 'mice_with_allele': 0}

PASS - section 4 reproduces from the raw profiles.
